In [1]:
# from RNN note
from torch import nn
import torch.nn.functional as F
# 检查是否支持MPS
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("使用MPS！")
else:
    device = torch.device("cpu")
    print("使用CPU")
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset
import matplotlib.pyplot as plt

# 使用通配符匹配所有分片
data_path = "/Users/hujinjia/PycharmProjects/JupyterProject/python/python/final/jsonl/train/*.jsonl.gz"
datasets = load_dataset('json', data_files=data_path)
datasets = datasets['train'].filter(lambda x: 'apache/spark' in x['repo'])
print(datasets[8]['original_string'])

class CharTokenizer:
    def __init__(self, data, end_ind=0):
        chars = sorted(list(set(''.join(data))))
        self.char2ind = {s: i + 1 for i, s in
                         enumerate(chars)}  # 创建 {字符: 索引} 的映射。self.char2ind = {'a': 2, 'b': 3, 'c': 4}
        self.char2ind['<|e|>'] = end_ind
        self.ind2char = {v: k for k, v in
                         self.char2ind.items()}  # 创建从索引到字符的反向映射，便于解码。self.ind2chat = {0: '<|b|>', 1: '<|e|>', 2: 'a', 3: 'b', 4: 'c'}
        self.end_ind = end_ind

    def encode(self, x):
        return [self.char2ind[i] for i in x]

    def decode(self, x):
        if isinstance(x, int):
            return self.ind2char[x]
        else:
            return [self.ind2char[i] for i in x]


tokenizer = CharTokenizer(datasets['original_string'])
#测试
test_str = 'def f(x)'
re = tokenizer.encode(test_str)
print(re)
''.join(tokenizer.decode(range(len(tokenizer.char2ind))))

使用MPS！


/Users/hujinjia/PycharmProjects/JupyterProject/.venv1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


def to_arrow_schema(schema):
    """ Convert a schema from Spark to Arrow
    """
    import pyarrow as pa
    fields = [pa.field(field.name, to_arrow_type(field.dataType), nullable=field.nullable)
              for field in schema]
    return pa.schema(fields)
[70, 71, 72, 2, 72, 10, 90, 11]


'<|e|>\n !"#$%&\'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\\]^_`abcdefghijklmnopqrstuvwxyz{|}~ö'

In [2]:
class RNN(nn.Module):

    def __init__(self, input_size, hidden_size):
        super(RNN, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.i2h = nn.Linear(input_size + hidden_size, hidden_size)

    def init_hidden(self, B,  device):
        return torch.zeros((B, self.hidden_size), device=device)

    def forward(self, input, hidden=None):
        # input:  (B,T,C)
        # hidden: (B,  H)
        # out:    (B,T,H)
        B, T, C = input.shape
        re = []
        if hidden is None:
            hidden = self.init_hidden(B, input.device)
        for i in range(T):
            combined = torch.concat((input[:, i, :], hidden), dim=1) #  (B, T, C) → (B, C) -> (B, C+H)
            hidden = F.relu(self.i2h(combined)) #(B,H)
            re.append(hidden)
        return torch.stack(re, dim=1) #(B,T,H)

In [3]:
"""
a = torch.zeros(3,4)
b = a + 1
c = torch.stack([a,b], dim=1)
c.shape, c[:, 0, :], c[:, 1, :]
"""

'\na = torch.zeros(3,4)\nb = a + 1\nc = torch.stack([a,b], dim=1)\nc.shape, c[:, 0, :], c[:, 1, :]\n'

In [4]:
"""
output1 = torch.randn(2, 5)
output2 = torch.randn(2, 5)
output3 = torch.randn(2, 5)
output = torch.stack((output1, output2, output3), dim=1)
print(output1)
print(output2)
print(output3)
print(output)
"""

'\noutput1 = torch.randn(2, 5)\noutput2 = torch.randn(2, 5)\noutput3 = torch.randn(2, 5)\noutput = torch.stack((output1, output2, output3), dim=1)\nprint(output1)\nprint(output2)\nprint(output3)\nprint(output)\n'

In [5]:
class CharRNNBatch(nn.Module):

    def __init__(self,vs):
        # vs：词汇表大小
        super().__init__()
        emb_size = 256 #每个字符被映射成256维的向量
        hidden_size = 128 #RNN内部隐藏状态的维度
        self.emb = nn.Embedding(vs, emb_size) #嵌入层（Embedding Layer），将离散的字符索引转换成连续的向量表示
        self.rnn1 = RNN(emb_size, hidden_size)  #输出维度: 128 (隐藏状态)，处理序列信息，捕捉上下文依赖
        self.ln1 = nn.LayerNorm(hidden_size)
        self.rnn2 = RNN(hidden_size, hidden_size) #第二层RNN，堆叠RNN层，增加模型容量和表达能力
        self.ln2 = nn.LayerNorm(hidden_size)
        self.lm = nn.Linear(hidden_size, vs)
        self.dp = nn.Dropout(0.4) #Dropout: 以0.4的概率随机丢弃神经元

    def forward(self, x):
        # x;(B,T) 。B：Batch size 。 T：每个样本包含多少个字符（或时间步）
        # 暂不实现初始隐藏状态的输入
        B = x.shape[0]
        embedding = self.emb(x)  #(B, T , emb_size)
        h = F.relu(self.ln1(self.rnn1(embedding)))  # (B, T, hidden_size)
        h = self.dp(h)
        h = F.relu(self.ln2(self.rnn2(h)))         # (B, T, hidden_size)
        h = self.dp(h)
        out = self.lm(h)                           #(B, T, VS)
        return out


In [6]:
a = tokenizer.char2ind
c_model =CharRNNBatch(len(a)).to(device)
print(c_model)

CharRNNBatch(
  (emb): Embedding(98, 256)
  (rnn1): RNN(
    (i2h): Linear(in_features=384, out_features=128, bias=True)
  )
  (ln1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (rnn2): RNN(
    (i2h): Linear(in_features=256, out_features=128, bias=True)
  )
  (ln2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (lm): Linear(in_features=128, out_features=98, bias=True)
  (dp): Dropout(p=0.4, inplace=False)
)


In [7]:
@torch.no_grad()
# 生成字
def generate(model, context,tokenizer,max_new_tokens=300):
    # context:(B,T), B =1
    # out = []
    out = context.tolist()[0] #将背景放到输入中
    model.eval()
    for _ in range(max_new_tokens):
        logits = model(context) #(B,T,VS) -> (1,T,98)
        probs = F.softmax(logits[:,-1,:], dim=-1) # 取最后一个logits (1, 98)
        #随机生成文本
        ix = torch.multinomial(probs, num_samples=1)  #(1,1)
        # 更新背景
        context = torch.concat((context, ix), dim=-1)
        out.append(ix.item())
        if out[-1] == tokenizer.end_ind:
            break
    model.train()
    return out

In [8]:
#测试generate
context = torch.tensor( tokenizer.encode('def'),device = device).unsqueeze(0)
print(''.join(tokenizer.decode(generate(c_model, context,tokenizer))))

def*lo%
N3[5`(1uSfu0#fieno<|e|>


In [9]:
def process(data, tokenizer, sequence_len):
    text = data['original_string']
    inputs, labels = [],[]
    for t in text:
        enc = tokenizer.encode(t)
        enc += [tokenizer.end_ind]
        for i in range(len(enc) - sequence_len):
            inputs.append(enc[i:i+sequence_len])
            labels.append(enc[i+1:i+1+sequence_len])
    return{'inputs': inputs, 'labels': labels}

In [10]:
tokenized = datasets.train_test_split(test_size = 0.1, seed = 1024, shuffle = True)
f = lambda x: process(x, tokenizer,sequence_len = 64)
tokenized = tokenized.map(f, batched=True, remove_columns = datasets.column_names)
tokenized.set_format(type='torch', device = device)

In [11]:
print(tokenized['train']['inputs'].shape)

torch.Size([605913, 64])


In [12]:
print(tokenized['train']['labels'].shape)

torch.Size([605913, 64])


In [13]:
train_loader = DataLoader(tokenized['train'], batch_size=1000, shuffle=True)
test_loader = DataLoader(tokenized['test'], batch_size=1000, shuffle=True)

In [14]:
next(iter(train_loader))

{'inputs': tensor([[67, 86, 69,  ..., 53, 71, 84],
         [75, 78, 70,  ...,  2,  2,  2],
         [ 2,  2,  2,  ..., 80, 10, 11],
         ...,
         [ 1,  1,  2,  ..., 85, 10, 11],
         [67, 91, 16,  ..., 86, 71, 90],
         [67, 79,  2,  ..., 87, 80, 83]], device='mps:0'),
 'labels': tensor([[86, 69, 74,  ..., 71, 84, 75],
         [78, 70, 82,  ...,  2,  2,  5],
         [ 2,  2,  2,  ..., 10, 11, 16],
         ...,
         [ 1,  2,  2,  ..., 10, 11, 28],
         [91, 16, 76,  ..., 71, 90, 86],
         [79,  2, 67,  ..., 80, 83, 87]], device='mps:0')}

In [15]:
eval_iters = 10

def estimate_loss(model):
    re = {} #创建一个空字典，用于存储训练损失和测试损失,最终返回的结构：{'train': 0.123, 'test': 0.456}
    model.eval() #将训练模式切换为评估模式
    train_loss = _loss(model, train_loader)
    test_loss = _loss(model, test_loader)
    re['train'] = train_loss
    re['test'] = test_loss
    model.train()  #切换回训练模式
    return re

@torch.no_grad()
def _loss(model, data_loader):
    #计算模型在不同数据集下面的评估指标
    loss = []
    data_iter = iter(data_loader)
    # 随机使用多个批量数据来评估模型效果
    for k in range(eval_iters):
        data = next(data_iter, None) #如果没有数据，返回 None
        if data is None: # 如果当前迭代器没有数据了
            data_iter = iter(data_loader) # 重新创建迭代器
            data = next(data_iter, None) # 获取第一个 batch
        inputs, labels = data['inputs'], data['labels'] # (B, T)
        logits = model(inputs)                          # (B, T, vs)
        # 官网交叉熵需要的纬度（B ，VS ，...)
        loss.append(F.cross_entropy(logits.transpose(-2,-1), labels).item()) # item() 将损失值(张量）转换为 Python 数字
    return torch.tensor(loss).mean().item()

estimate_loss(c_model)

{'train': 4.7308220863342285, 'test': 4.735166549682617}

In [16]:
def train_model(model,optimizer,epochs=10):
    lossi = [] # 记录模型在训练集上的模型损失
    for epoch in range(epochs):
        for i, data in enumerate(train_loader, 0):
            inputs, labels = data['inputs'], data['labels'] # (B, T)
            optimizer.zero_grad()
            logits = model(inputs)                          # (B ,T vs)
            loss = F.cross_entropy(logits.transpose(-2,-1), labels)          # 如上
            lossi.append(loss.item())
            loss.backward()
            optimizer.step()
        # 评估模型，并输出结果
        stats = estimate_loss(model)
        train_loss = f'train loss {stats['train']:.4f}'
        test_loss = f'test loss {stats['test']:.4f}'
        print(f'epoch {epoch:>2}: {train_loss}, {test_loss}')
    return lossi

In [17]:
learning_rate = 0.01
l = train_model(c_model, optim.Adam(c_model.parameters(), lr=learning_rate))

epoch  0: train loss 1.3765, test loss 1.4963
epoch  1: train loss 1.3162, test loss 1.4272
epoch  2: train loss 1.2949, test loss 1.4210
epoch  3: train loss 1.2682, test loss 1.3991
epoch  4: train loss 1.2688, test loss 1.4000
epoch  5: train loss 3.1635, test loss 3.1964
epoch  6: train loss 3.1684, test loss 3.2003
epoch  7: train loss 3.1699, test loss 3.1862
epoch  8: train loss 3.1650, test loss 3.1972
epoch  9: train loss 3.1552, test loss 3.1958


In [18]:
context = torch.tensor( tokenizer.encode('def'),device = device).unsqueeze(0)
print(''.join(tokenizer.decode(generate(c_model, context,tokenizer))))

defho .eelt  m n` ueuts s( e':i tir ec r  fr,, pa(  ki mPy v i _avoi  ng sasn"  v eTN erg .   cm)raitr  s  xIpt  ao]l ec sp rmesn e   se  ee  rdn  ese
     s h crr clnnca aaabrrnetr  d s     e  p ip a eAcant a z|e  o -bv tDen_ sa
t2   eRehCa f m_u m hs fretcdt  a`SAsepi" e ow "tot uee)a v( y i aag , sl
